# Notebook 1 — Introducción a la Google Cloud Vision API

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lección 1:** Introducción

Este notebook acompaña al manual *Google Vision API for Dummies*. Aquí encontrarás:

- Explicaciones breves de cada concepto (ya desarrollado a fondo en el manual)
- Celdas de código con ejemplos ejecutables usando la API real de Google Vision
- Ejercicios con celdas para que completes tú el código, seguidas de una solución desplegable (haz clic en **Ver solución** para comprobar tu propuesta)

> **Antes de ejecutar el código real de este notebook** necesitas un proyecto de Google Cloud con la Vision API habilitada y un archivo de credenciales JSON. Si todavía no lo tienes, abre primero el notebook `02_Configuracion_y_Setup.ipynb` y vuelve después — aquí puedes seguir leyendo y estudiando la teoría sin problema.

## 1. ¿Qué es la Google Cloud Vision API?

La inteligencia artificial (IA) permite que las máquinas ejecuten tareas que antes exigían inteligencia humana. La visión por computador es una de sus aplicaciones más visibles: la capacidad de interpretar datos visuales. Google pone esa capacidad a tu alcance con la **Google Cloud Vision API**, un servicio en la nube que analiza imágenes usando modelos de aprendizaje automático ya entrenados, sin que tengas que entrenar nada tú mismo.

Envías una imagen, te autenticas con tus credenciales, y recibes una respuesta estructurada en formato JSON (notación de objetos de JavaScript).

**Qué puede detectar la API (lo irás viendo lección a lección en este curso):**

- **Etiquetas**: objetos, escenas y conceptos ("perro", "árbol", "coche")
- **Rostros**: posición de ojos, nariz y boca, y probabilidad de emociones (alegría, ira, sorpresa, tristeza)
- **Texto (OCR)**: reconocimiento óptico de caracteres, impreso o manuscrito
- **Objetos**: ubicación precisa de cada objeto dentro de la imagen
- **Monumentos, logotipos y contenido explícito**: funciones avanzadas (lección 6)

### El ejemplo del robot Gopi

El material original del curso propone un caso muy ilustrativo: un robot Raspberry Pi con cámara, llamado *Gopi*. Gopi no ejecuta el modelo de IA en su propio hardware — envía la foto que capturó a la nube, la Vision API la analiza, y le devuelve el resultado. Con detección de rostros y emociones, Gopi puede acercarse a una cara sonriente o alejarse ante ira o sorpresa. Lo recrearás en el ejercicio 2.

## 2. Vision API frente a AutoML Vision API

Dentro de "Vision AI", Google separa dos servicios que conviene diferenciar antes de escribir código:

| | **Vision API** | **AutoML Vision API** |
|---|---|---|
| Modelo | Preentrenado por Google | Entrenas tu propio modelo con tus imágenes |
| Interfaz | Código (API REST o cliente Python) | Interfaz gráfica para subir y etiquetar imágenes |
| Cuándo elegirla | Te basta el reconocimiento genérico ya entrenado | Necesitas categorías muy específicas de tu negocio |
| Coste | Pago por uso | Coste adicional por entrenamiento |

En este curso trabajarás siempre con la **Vision API** (el modelo preentrenado), a través de la librería cliente de Python `google-cloud-vision`.

## 3. Preparar el cliente de Python

La celda siguiente crea el cliente que usarás durante todo el notebook. Si ya configuraste la variable de entorno `GOOGLE_APPLICATION_CREDENTIALS` (lo verás en el notebook 2), esta celda debería ejecutarse sin errores.

In [ ]:
import os

# Ajusta esta ruta a tu archivo de credenciales JSON (ver notebook 02)
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision

try:
    client = vision.ImageAnnotatorClient()
    print("Cliente creado correctamente.")
except Exception as e:
    print("No se pudo crear el cliente. Revisa tus credenciales:", e)


## Ejercicio 1 — Detección de etiquetas básica

Completa la función `detectar_etiquetas` para que:

1. Abra la imagen indicada en `image_path` en modo binario
2. Construya un objeto `vision.Image` con su contenido
3. Llame a `client.label_detection(...)`
4. Devuelva una lista de tuplas `(descripcion, score)` ordenada de mayor a menor confianza

Usa cualquier imagen propia (por ejemplo, una foto de un animal, un paisaje o un objeto de tu escritorio) guardada en la misma carpeta que este notebook.

In [ ]:
def detectar_etiquetas(image_path):
    # TODO 1: abre el archivo en modo binario y lee su contenido
    content = ...

    # TODO 2: crea el objeto vision.Image
    image = ...

    # TODO 3: llama al método de detección de etiquetas
    response = ...

    # TODO 4: construye y devuelve la lista (descripcion, score) ordenada por score descendente
    etiquetas = ...
    return etiquetas


# Descomenta y ajusta la ruta cuando tengas una imagen propia:
# resultado = detectar_etiquetas("mi_imagen.jpg")
# for descripcion, score in resultado:
#     print(f"{descripcion}: {score:.2f}")


<details>
<summary><strong>Ver solución</strong></summary>

```python
def detectar_etiquetas(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.label_detection(image=image)

    etiquetas = [(label.description, label.score) for label in response.label_annotations]
    etiquetas.sort(key=lambda x: x[1], reverse=True)
    return etiquetas


resultado = detectar_etiquetas("mi_imagen.jpg")
for descripcion, score in resultado:
    print(f"{descripcion}: {score:.2f}")
```

</details>

## Ejercicio 2 — Simulación del robot Gopi

Recrea la lógica de decisión de Gopi: dada una imagen con un rostro, el robot debe decidir si "acercarse" (si detecta alegría con probabilidad alta) o "alejarse" (si detecta ira o sorpresa con probabilidad alta).

Pistas:

- Usa `client.face_detection(image=image)` y toma el primer rostro de `response.face_annotations`
- Los campos `joy_likelihood`, `anger_likelihood` y `surprise_likelihood` devuelven un entero que corresponde a la enumeración `Likelihood`: `UNKNOWN` (0), `VERY_UNLIKELY` (1), `UNLIKELY` (2), `POSSIBLE` (3), `LIKELY` (4), `VERY_LIKELY` (5)
- Considera "probabilidad alta" cualquier valor igual o superior a `LIKELY` (4)

In [ ]:
def decision_gopi(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.face_detection(image=image)

    if not response.face_annotations:
        return "No se detectó ningún rostro."

    rostro = response.face_annotations[0]

    # TODO 1: define el umbral "alto" (pista: el valor entero de LIKELY)
    umbral_alto = ...

    # TODO 2: decide la acción de Gopi según joy_likelihood, anger_likelihood y surprise_likelihood
    if ...:
        return "Gopi se acerca (detectó alegría)"
    elif ...:
        return "Gopi se aleja (detectó ira o sorpresa)"
    else:
        return "Gopi se queda quieto (emoción no concluyente)"


# print(decision_gopi("cara.jpg"))


<details>
<summary><strong>Ver solución</strong></summary>

```python
def decision_gopi(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.face_detection(image=image)

    if not response.face_annotations:
        return "No se detectó ningún rostro."

    rostro = response.face_annotations[0]
    umbral_alto = 4  # vision.Likelihood.LIKELY

    if rostro.joy_likelihood >= umbral_alto:
        return "Gopi se acerca (detectó alegría)"
    elif rostro.anger_likelihood >= umbral_alto or rostro.surprise_likelihood >= umbral_alto:
        return "Gopi se aleja (detectó ira o sorpresa)"
    else:
        return "Gopi se queda quieto (emoción no concluyente)"


print(decision_gopi("cara.jpg"))
```

</details>

## Ejercicio 3 — Explora el cliente

Con el cliente ya creado, ejecuta `dir(client)` y anota (en la celda Markdown siguiente, editándola tú) al menos cinco métodos que reconozcas o que te llamen la atención de cara a las próximas lecciones.

In [ ]:
# Ejecuta esta celda y explora el resultado
print([m for m in dir(client) if not m.startswith("_")])


**Tus notas** (edita esta celda):

1. `label_detection` — ...
2. `text_detection` — ...
3. `face_detection` — ...
4. `safe_search_detection` — ...
5. `image_properties` — ...

---

**Siguiente paso:** abre `02_Configuracion_y_Setup.ipynb` para dejar tu entorno completamente configurado, o `03_Texto_en_Imagenes.ipynb` si ya lo tienes listo.